# Geopack Python SDK: Getting Started

Welcome to the Geopack Geoportal v2 Python SDK. This notebook will guide you through the basics of connecting to the portal, authenticating, and exploring available datasets.

---

### 🚀 Quick Setup

#### 1. Environment & Dependencies
You have two ways to prepare your environment:

*   **Option A: Inside Notebook (Easiest)**
    Run the cell below (under "Setup Dependencies") to install everything directly into your current kernel.
*   **Option B: Manual Virtual Environment (Recommended for Clean Setup)**
    Open your terminal in the `python-sdk` folder and run:
    ```bash
    python -m venv venv
    venv\Scripts\activate  # On Windows
    source venv/bin/activate  # On Linux/macOS
    pip install geopandas matplotlib python-dotenv
    ```

#### 2. SDK Installation Options
*   **Method A (Local Source)**: This notebook is pre-configured to use the `src/` folder directly. No installation needed if you are in the repository!
*   **Method B (Registry)**: `pip install geopack-sdk`

#### 3. Credentials
Ensure you have a `.env` file in this `notebooks/` folder with your `GEOPACK_API_URL`, `GEOPACK_USERNAME`, and `GEOPACK_PASSWORD`.

---

In [1]:
# Setup Dependencies
# Run this cell to install required libraries if you haven't already
%pip install python-dotenv geopandas matplotlib


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Initialize the Client
First, we import the `GeopackClient` and initialize it with the API URL.

In [2]:
# 1. Setup & SDK Import
# Enable auto-reload for development
%load_ext autoreload
%autoreload 2

import os
import os
import sys
from dotenv import load_dotenv

# --- SMART SOURCE IMPORT ---
# This ensures we use the local 'src' directory even if the package isn't installed.
# It calculates the path relative to this notebook's location.
try:
    current_dir = os.getcwd()
    source_path = os.path.abspath(os.path.join(current_dir, "..", "src"))
    
    if os.path.exists(source_path):
        if source_path not in sys.path:
            sys.path.insert(0, source_path)
        print(f"ℹ️ Using SDK from local source: {source_path}")
    else:
        print("ℹ️ Using SDK from installed site-packages (pip)")
except Exception:
    print("⚠️ Could not determine local source path, falling back to pip.")

from geopack_sdk import GeopackClient

# --- Load Configuration ---
load_dotenv()
API_URL = os.getenv("GEOPACK_API_URL", "http://localhost:3000/api")
client = GeopackClient(base_url=API_URL)

print(f"✅ Client initialized for: {API_URL}")


ℹ️ Using SDK from local source: d:\Works\geopack-geoportal\geopack-geoportal-v2\python-sdk\src
✅ Client initialized for: http://localhost:3000/api


## 2. Authentication
Log in using your credentials to obtain a JWT token.

In [3]:
import os
USERNAME = os.getenv("GEOPACK_USERNAME", "admin")
PASSWORD = os.getenv("GEOPACK_PASSWORD", "password")

try:
    client.auth.login(USERNAME, PASSWORD)
    print("✅ Login successful!")
except Exception as e:
    print(f"❌ Login failed: {e}")


✅ Login successful!


## 3. Explore Datasets
Now let's list the available datasets in the portal.

In [4]:
datasets = client.datasets.list(page_size=50)

print(f"Found {len(datasets.datasets)} datasets:\n")
for ds in datasets.datasets:
    print(f"- [{ds.id}] {ds.name}  ({ds.dataType or 'N/A'} {ds.subType or ''}) [{ds.dataStore.name if ds.dataStore else 'N/A'}]")

Found 50 datasets:

- [2399] hillshade12  (raster SingleBand) [Default Filesystem GDB]
- [2398] Rural_District.shp  (vector MultiPolygon) [Default Filesystem GDB]
- [2397] Rural_District.shp  (vector MultiPolygon) [Default Filesystem GDB]
- [2396] Rural_District.shp  (vector MultiPolygon) [Default Filesystem GDB]
- [2395] hillshade12  (raster SingleBand) [Default Filesystem GDB]
- [2394] hillshade12  (raster SingleBand) [Default Filesystem GDB]
- [2393] hillshade12  (raster SingleBand) [Default Filesystem GDB]
- [2392] hillshade12  (raster SingleBand) [Default Filesystem GDB]
- [2391] hillshade12  (raster SingleBand) [Default Filesystem GDB]
- [2390] hillshade12  (raster SingleBand) [Default Filesystem GDB]
- [2389] hillshade12  (raster SingleBand) [Default Filesystem GDB]
- [2388] hillshade12  (raster SingleBand) [Default Filesystem GDB]
- [2387] hillshade12  (raster SingleBand) [Default Filesystem GDB]
- [2386] Rural_District.shp  (vector MultiPolygon) [Default Filesystem GDB]
- [238

## 4. Advanced Filtering
The SDK supports the same powerful filtering as the portal. Let's find only the **Vector** datasets.

In [5]:
# Using the same filter structure as the Portal UI
vector_only = client.datasets.list(active_filters={"dataType": "vector"})

print(f"Found {len(vector_only.datasets)} vector datasets.")

Found 10 vector datasets.


## 5. Get Detailed Metadata
Retrieve full information about a specific dataset, including its SRS, extent, and statistics.

In [6]:
if datasets.datasets:
    target_id = datasets.datasets[0].id
    details = client.datasets.get(target_id)
    
    import json
    print(f"Detailed info for dataset #{target_id}:")
    print(json.dumps(details.model_dump(mode='json'), indent=2, default=str))
else:
    print("No datasets available to inspect.")

Detailed info for dataset #2399:
{
  "id": 2399,
  "name": "hillshade12",
  "description": "sdf (Source: hillshade_1_output.tif)",
  "dataType": "raster",
  "subType": "SingleBand",
  "keywords": "",
  "ownerUserId": 1,
  "workgroupId": 1,
  "dataStoreId": 11,
  "details": "{\"type\":\"raster\",\"source\":{\"driver\":\"COG\",\"container\":\"app_data/gdb/default\",\"datasetFolder\":\"61909311-8722-4720-8422-99e06815ae1d\",\"originalFileName\":\"hillshade_1_output.tif\",\"fileType\":\".tif\",\"originalFilePath\":\"original\\\\hillshade_1_output.tif\",\"cogPath\":\"processed\\\\hillshade_1_output_cog.tif\",\"mbtilesPath\":null,\"isInPlace\":false},\"spatialReference\":{\"srid\":3857,\"wkt\":\"PROJCS[\\\"WGS 84 / Pseudo-Mercator\\\",\\r\\n    GEOGCS[\\\"WGS 84\\\",\\r\\n        DATUM[\\\"WGS_1984\\\",\\r\\n            SPHEROID[\\\"WGS 84\\\",6378137,298.257223563,\\r\\n                AUTHORITY[\\\"EPSG\\\",\\\"7030\\\"]],\\r\\n            AUTHORITY[\\\"EPSG\\\",\\\"6326\\\"]],\\r\\n      